**DEMO OF standard gridder**

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/imaging/demo_standard_grid.ipynb)

# Demo of the standard gridder

This notebook shows how to grid interferometer visibilities into a dirty image with the **current AstroVIPER processing-function API**.

**Part A - recover known sources.** We build a sky model containing a few point sources, degrid it onto the *uv* coverage of a real measurement set to synthesize visibilities, then grid those visibilities back into a dirty image and check that the sources are recovered at the right positions and (approximately) the right flux.

**Part B - compare with CASA.** Using the model visibilities of an NGC 5921 measurement set, we make continuum and cube dirty images and compare them with the equivalent images produced by CASA.

## Assumptions
The centre pixel is assumed to be the phase centre.

## API update

This tutorial was rewritten to the current processing-function API. The old low-level helpers (`generate_ms4_with_point_sources`, `standard_grid_numpy_wrap_input_checked`, `create_prolate_spheroidal_kernel`, `grid2image_spheroid_ms4`, `ifft_uv_to_lm`) have been removed. Dirty / residual images are now made from an `xradio` image dataset with:

* `create_prolate_spheroidal_kernel_1D` - the 1-D gridding convolution kernel.
* `calculate_imaging_weights` - imaging weights (`natural`, `uniform`, `briggs`).
* `make_undeconvolved_image_single_field` (or the lower-level `add_visibility_grid_single_field`) - grid `VISIBILITY` / `VISIBILITY_NORMALIZATION` into the image dataset.
* `ifft_norm_img_xds` - inverse FFT + gridding correction + normalization to the sky image.
* `get_visibility_grid_single_field` / `fft_norm_img_xds` - degrid a sky model into model visibilities (used in Part A to synthesize data).

The empty image is created with `xradio.image.make_empty_sky_image` and carries a `residual` data group, exactly as in `make_psf_demo.ipynb`.

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    os.system("pip install --upgrade astroviper")

    import astroviper

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

## API

In [ ]:
import numpy as np
import xarray as xr
from astropy import units as u
from matplotlib import pylab as pl
from xradio.image import make_empty_sky_image
from xradio.measurement_set import load_processing_set

from astroviper.processing_functions.imaging.add_visibility_grid import (
    add_visibility_grid_single_field,
)
from astroviper.processing_functions.imaging.calculate_imaging_weights import (
    calculate_imaging_weights,
)
from astroviper.processing_functions.imaging.fft_normalize_prolate_spheriodal_gridder import (
    fft_norm_img_xds,
    ifft_norm_img_xds,
)
from astroviper.processing_functions.imaging.get_visibility_grid import (
    get_visibility_grid_single_field,
)
from astroviper.processing_functions.imaging.gridding_convolution_functions.gcf_prolate_spheroidal import (
    create_prolate_spheroidal_kernel_1D,
)
from astroviper.processing_functions.imaging.make_undeconvolved_image import (
    make_undeconvolved_image_single_field,
)

In [ ]:
help(make_undeconvolved_image_single_field)

In [ ]:
help(get_visibility_grid_single_field)

## Download data

We download an NGC 5921 measurement set (a CASA model gridded into an MS v4 processing set) together with the continuum and cube images CASA made from the same data.

In [ ]:
!pip install gdown
import os

import gdown

# Measurement set (a CASA model gridded into an MS v4). Skip the download if it
# is already present locally.
if not os.path.exists("lala.zip"):
    gdown.download(id="19br3EYwdtu82iF4JkRaX-9u2_bhNAMjJ", output="lala.zip")
!unzip -o -q lala.zip
# Continuum and cube images made in CASA (for comparison below).
if not os.path.exists("cube_image.npy"):
    gdown.download(id="1hHLUB9uJT9okRpYHHHBu4WUXcsOW6Sk2", output="cube_image.npy")
if not os.path.exists("cont_image.npy"):
    gdown.download(id="1tFHACLA-5qnpz76HG2UbixakqSX20MIH", output="cont_image.npy")

## Load the processing set and set up imaging

Most gridding parameters come from the measurement-set metadata. We build the prolate-spheroidal kernel (oversampling 100, support 7), read the phase centre, frequency axis and polarization axis, choose a 256x256 image with 15 arcsec cells, and compute natural imaging weights (`WEIGHT_IMAGING`), which the gridder needs.

In [ ]:
incr = (15 * u.arcsec).to("rad").value
image_size = [256, 256]
cell_size = [-incr, incr]  # RA increment is negative by convention

ngc_xdt = load_processing_set("ngc5921_casa_model.ps.zarr")
ms_name = list(ngc_xdt.keys())[0]

# Phase centre, frequency axis and polarization axis come from the processing set.
combined = ngc_xdt.xr_ps.get_combined_field_and_source_xds()
phase_center = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
frequency_coords = ngc_xdt.xr_ps.get_freq_axis().values
pol_coords = list(ngc_xdt[ms_name].polarization.values)
n_freq = len(frequency_coords)
print("polarizations:", pol_coords, " n_freq:", n_freq)

# 1-D prolate-spheroidal gridding convolution kernel (oversampling 100, support 7).
cgk_1D = create_prolate_spheroidal_kernel_1D(100, 7)

image_params = {
    "image_size": image_size,
    "cell_size": cell_size,
    "fft_padding": 1.2,
    "cpp_gridder": True,
}


def empty_image(frequency_coords):
    """Empty xradio sky image (correlation basis) matching the processing set."""
    img_xds = make_empty_sky_image(
        phase_center=phase_center,
        image_size=image_size,
        cell_size=cell_size,
        frequency_coords=frequency_coords,
        pol_coords=pol_coords,
        time_coords=[0],
        do_sky_coords=True,
    )
    img_xds.attrs["type"] = "image_dataset"
    return img_xds

In [ ]:
# Natural imaging weights on the 'base' data group (needed by the gridder below).
calculate_imaging_weights(
    ngc_xdt,
    empty_image(frequency_coords),  # geometry only; natural weighting ignores it
    imaging_weights_params={
        "weighting": "natural",
        "casa_weighting_implementation": True,
    },
    ms_data_group_in_name="base",
    ms_data_group_out_name="base",
    ms_data_group_out_modified={"weight_imaging": "WEIGHT_IMAGING"},
)

# Part A - recover known point sources

To make a controlled test we put a few point sources of known position and flux into a `SKY_MODEL` image, transform it to the *uv* plane with `fft_norm_img_xds`, and degrid it onto the measurement set's *uv* coverage with `get_visibility_grid_single_field` to synthesize visibilities (`VISIBILITY_SYNTH`). Gridding those visibilities back and inverse-transforming should recover the sources. We put the same flux in both parallel-hand correlations, so each source is an unpolarized Stokes I source.

In [ ]:
# Point sources: (l pixel, m pixel, flux in Jy).
sources = [(128, 128, 1.0), (90, 160, 0.7), (170, 70, 0.5)]

model_img = empty_image(frequency_coords)
sky_model = np.zeros((1, n_freq, len(pol_coords), image_size[0], image_size[1]))
for l_pix, m_pix, flux in sources:
    sky_model[0, :, :, l_pix, m_pix] = (
        flux  # same flux in both correlations -> Stokes I
    )
model_img["SKY_MODEL"] = xr.DataArray(
    sky_model, dims=("time", "frequency", "polarization", "l", "m")
)
model_img = model_img.xr_img.add_data_group(
    new_data_group_name="model",
    new_data_group={"sky": "SKY_MODEL", "description": "point sources", "date": "2026"},
)

# Sky model image -> model uv grid (VISIBILITY_MODEL).
model_img = fft_norm_img_xds(
    model_img,
    image_params=image_params,
    image_data_group_in_name="model",
    image_data_group_out_name="model",
    image_data_group_out_modified={"visibility": "VISIBILITY_MODEL"},
    data_variables_to_process=["sky"],
    processing_function_threads=1,
    fft_backend="scipy",
    complex_dtype=np.complex128,
)

In [ ]:
# Degrid the model uv grid onto the MS uv coverage -> synthetic visibilities.
for _, ms_xdt in ngc_xdt.items():
    get_visibility_grid_single_field(
        ms_xdt,
        cgk_1D,
        model_img,
        ms_data_group_in_name="base",
        ms_data_group_out_name="synthetic",
        ms_data_group_out_modified={"correlated_data": "VISIBILITY_SYNTH"},
        image_data_group_in_name="model",
        overwrite=True,
        chan_mode="cube",
        fft_padding=image_params["fft_padding"],
        processing_function_threads=1,
    )

In [ ]:
# Grid the synthetic visibilities back into a dirty image, then transform to lm.
recovered_img = empty_image(frequency_coords)
recovered_img = recovered_img.xr_img.add_data_group(
    new_data_group_name="residual",
    new_data_group={"description": "recovered", "date": "2026"},
)
recovered_img, _ = make_undeconvolved_image_single_field(
    ngc_xdt,
    recovered_img,
    image_params,
    cgk_1D,
    True,  # is_n_iter_0: dirty image (no model subtraction)
    ms_data_group_in_name="synthetic",
    image_data_group_out_name="residual",
    processing_function_threads=1,
    complex_dtype=np.complex128,
)
recovered_img = ifft_norm_img_xds(
    recovered_img,
    image_params=image_params,
    image_data_group_in_name="residual",
    image_data_group_out_name="residual",
    image_data_group_out_modified={"sky": "SKY_RESIDUAL"},
    processing_function_threads=1,
    fft_backend="scipy",
    complex_dtype=np.complex128,
)

# Average the parallel-hand correlations (-> Stokes I) and the channels (-> continuum).
recovered = recovered_img["SKY_RESIDUAL"].values[0].mean(axis=(0, 1))  # (l, m)

In [ ]:
print("Recovered vs input sources (dirty-image value at each input pixel):")
for l_pix, m_pix, flux in sources:
    print(
        f"  source at (l={l_pix}, m={m_pix}) input {flux:.2f} Jy"
        f"  ->  recovered {recovered[l_pix, m_pix]:.3f} Jy/beam"
    )

In [ ]:
pl.imshow(recovered)
pl.colorbar()
pl.title("Part A: recovered dirty image (Stokes I)")

# Part B - comparison with CASA

Now we grid the model visibilities (`VISIBILITY`) of the NGC 5921 measurement set into continuum and cube dirty images and compare them with the images CASA produced from the same data.

## Continuum image

For a continuum image we grid every channel onto a single-channel image (`chan_mode="continuum"`) with `add_visibility_grid_single_field`, then transform and normalize with `ifft_norm_img_xds`. The data has two correlations (RR, LL), so we average them for a poor man's Stokes I.

In [ ]:
# Single-channel continuum image at the mean frequency of the band.
cont_img = empty_image([float(np.mean(frequency_coords))])
cont_img = cont_img.xr_img.add_data_group(
    new_data_group_name="residual",
    new_data_group={"description": "continuum", "date": "2026"},
)
for _, ms_xdt in ngc_xdt.items():
    add_visibility_grid_single_field(
        ms_xdt,
        cgk_1D,
        cont_img,
        ms_data_group_in_name="base",
        image_data_group_in_name="residual",
        image_data_group_out_name="residual",
        image_data_group_out_modified={
            "visibility": "VISIBILITY",
            "visibility_normalization": "VISIBILITY_NORMALIZATION",
        },
        overwrite=True,
        chan_mode="continuum",
        fft_padding=image_params["fft_padding"],
        processing_function_threads=1,
    )
cont_img = ifft_norm_img_xds(
    cont_img,
    image_params=image_params,
    image_data_group_in_name="residual",
    image_data_group_out_name="residual",
    image_data_group_out_modified={"sky": "SKY_RESIDUAL"},
    processing_function_threads=1,
    fft_backend="scipy",
    complex_dtype=np.complex128,
)
# poor man's corr2Stokes: average RR and LL -> Stokes I  (time=0, channel=0)
cont_dirty = cont_img["SKY_RESIDUAL"].values[0, 0].mean(axis=0)  # (l, m)

In [ ]:
pl.imshow(cont_dirty)
pl.colorbar()
pl.title("AstroVIPER continuum dirty image")

In [ ]:
casa_cont = np.load("cont_image.npy")
pl.imshow(casa_cont)
pl.colorbar()
pl.title("CASA continuum image")

In [ ]:
# CASA edges are masked, so compare the interior only.
pl.imshow(casa_cont[10:246, 10:246] - cont_dirty[10:246, 10:246])
pl.colorbar()
pl.title("CASA - AstroVIPER (continuum)")

## Cube image

For a cube we use the high-level `make_undeconvolved_image_single_field`, which grids each channel onto its own image channel (`chan_mode="cube"`), followed by `ifft_norm_img_xds`.

In [ ]:
cube_img = empty_image(frequency_coords)
cube_img = cube_img.xr_img.add_data_group(
    new_data_group_name="residual",
    new_data_group={"description": "cube", "date": "2026"},
)
cube_img, _ = make_undeconvolved_image_single_field(
    ngc_xdt,
    cube_img,
    image_params,
    cgk_1D,
    True,
    ms_data_group_in_name="base",
    image_data_group_out_name="residual",
    processing_function_threads=1,
    complex_dtype=np.complex128,
)
cube_img = ifft_norm_img_xds(
    cube_img,
    image_params=image_params,
    image_data_group_in_name="residual",
    image_data_group_out_name="residual",
    image_data_group_out_modified={"sky": "SKY_RESIDUAL"},
    processing_function_threads=1,
    fft_backend="scipy",
    complex_dtype=np.complex128,
)
# Correlations to Stokes: average RR and LL over the polarization axis.
cube_dirty = cube_img["SKY_RESIDUAL"].values[0].mean(axis=1)  # (freq, l, m)

A little function to view a cube in a notebook. The original tutorial used an interactive `ipywidgets` channel slider; here we show a static montage of a few representative channels instead, so the notebook also runs headlessly (e.g. in CI / `nbconvert`). To get the interactive slider back, replace the body with an `ipywidgets.interact` over the channel index.

In [ ]:
def cube_view(cube, n_show=6):
    """Show a montage of a few representative channels of a cube.

    ``cube`` has shape (n_chan, l, m).  A static montage is used (rather than an
    interactive ipywidgets slider) so the notebook also runs in a headless
    context such as nbconvert / CI.
    """
    nchan = cube.shape[0]
    idx = np.unique(np.linspace(0, nchan - 1, min(n_show, nchan)).astype(int))
    ncol = len(idx)
    fig, axes = pl.subplots(1, ncol, figsize=(2.6 * ncol, 3.0))
    if ncol == 1:
        axes = [axes]
    for ax, ci in zip(axes, idx):
        im = ax.imshow(cube[ci, :, :], cmap="viridis")
        ax.set_title(f"channel {ci}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    pl.tight_layout()
    pl.show()

Let's look at the image produced

In [ ]:
cube_view(cube_dirty)

Similar image made in casa

In [ ]:
casa_cube = np.moveaxis(np.load("cube_image.npy"), -1, 0)
cube_view(casa_cube)

Let's look at the difference. The CASA edges are masked, so we avoid those in the difference.

In [ ]:
cube_view(casa_cube[:, 25:231, 25:231] - cube_dirty[:, 25:231, 25:231])